### RFE and Mutual Information to choose optimal variable for the dataset (and study potentially redundant ones)

In [5]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression, mutual_info_classif
from sklearn.model_selection import train_test_split
import seaborn as sns

# load data
df = pd.read_csv("../lifestyle_sustainability_data_processed.csv")

# some missing values, let's drop them for now
df = df.dropna()

# change this to your own target variable
TARGET_VARIABLE = "Rating"

### RFE - recursive feature elimination

In [6]:
# typical X/y -split
X = df.drop(TARGET_VARIABLE, axis=1)
y = df[TARGET_VARIABLE]

# define model (linear regression, random forest, XGBoost etc.)
# technically you can use pretty much any classic ML algorithm
# model = LinearRegression()

# idea: you might want to try RFE with multuple ML algorithms
# and then cross-validate the results (which results are common in all models)

# another idea: if you plan on using e.g. XGBoost in your final model
# it's probably a good idea to use the same XGBoost here
model = RandomForestRegressor()

# create RFE, place the model and choose number of optimal variables
rfe = RFE(estimator=model, n_features_to_select=7)

# fit the RFE model with our data
rfe.fit(X, y)

# get rankings and results
rankings = rfe.ranking_
support = rfe.support_

# build a DataFrame to wrap up result for easier inspection
results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

# you can use these results with any other knowledge you have
# from other optimal variable selection tools, and cross-validation
results_df

,Feature,Ranking,Selected
0,Age,1,True
2,HomeSize,1,True
6,CommunityInvolvement,1,True
5,EnvironmentalAwareness,1,True
7,MonthlyElectricityConsumption,1,True
8,MonthlyWaterConsumption,1,True
9,UsingPlasticProducts,1,True
1,LocalFoodFrequency,2,False
14,DietType_Balanced,3,False
28,DisposalMethods_Landfill,4,False


In [7]:
# the selection loosely follow the correlations it seems
# this might imply the data is more or less linear or something else
df.corr()[TARGET_VARIABLE].sort_values(ascending=False)

Rating                               1.000000
CommunityInvolvement                 0.501605
PhysicalActivities                   0.483083
EnvironmentalAwareness               0.473345
LocalFoodFrequency                   0.415085
SustainableBrands                    0.409035
DisposalMethods_Composting           0.405349
Gender_Female                        0.242502
TransportationMode_Bike              0.115145
Location_Suburban                    0.065974
HomeType_House                       0.063537
Gender_Non-Binary                    0.048399
Location_Urban                      -0.005787
HomeSize                            -0.031943
Location_Rural                      -0.064522
Age                                 -0.064904
TransportationMode_Public Transit   -0.140026
HomeType_Apartment                  -0.145324
DisposalMethods_Combination         -0.146984
EnergySource_Mixed                  -0.189693
DietType_Mostly Animal-Based        -0.216475
Gender_Male                       

### Mutual information - another alternative

In [8]:
# fit the mutual information algorithm
# => that's why either or works
mi = mutual_info_regression(X, y)

# convert results into DataFrame
mi_results = pd.Series(mi, index=X.columns).sort_values(ascending=False)

# high value => variable is strongly connected to target variable
# low value => weak connection
# e.g. very similar to phik-matrix
mi_results

MonthlyElectricityConsumption        0.483197
MonthlyWaterConsumption              0.406611
EnvironmentalAwareness               0.271828
CommunityInvolvement                 0.260142
HomeSize                             0.259239
PhysicalActivities                   0.236796
Age                                  0.227998
UsingPlasticProducts                 0.222640
ClothingFrequency                    0.176987
LocalFoodFrequency                   0.151895
SustainableBrands                    0.145101
EnergySource_Mixed                   0.141420
DietType_Mostly Animal-Based         0.112964
TransportationMode_Public Transit    0.106535
TransportationMode_Bike              0.101352
DisposalMethods_Composting           0.097880
EnergySource_Non-Renewable           0.092562
DisposalMethods_Landfill             0.087657
DietType_Balanced                    0.074179
TransportationMode_Car               0.059546
DisposalMethods_Combination          0.058854
Location_Rural                    